In [1]:
!pip install -q --upgrade \
    pdfplumber \
    tqdm

!apt-get install -y poppler-utils

!pip install opencv-python pdf2image numpy


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


# Đại Việt Sử Ký Toàn Thư

In [2]:
import re
import json
import pdfplumber
from tqdm import tqdm
import cv2
import numpy as np
from pdf2image import convert_from_path

PDF_PATH      = "DVSK_NhaTran.pdf"
OUTPUT_JSON   = "DVSKTT_Chunks.json"
BOOK_NAME     = "Đại Việt Sử Ký Toàn Thư"

In [3]:

SEPARATORS = [
    "\n\n",
    "\n",
    ". ",
    "! ", "? ", "; ", ", ", " ", "",
]

PAGE_MARKER_FMT = "\n[[[PAGE:{page}]]]\n"

HEADER_FOOTER_PATTERNS = [
    re.compile(r'^\s*\d{1,4}\s+Đại Việt Sử Ký Toàn Thư.*$', re.MULTILINE | re.IGNORECASE),
    re.compile(r'^Đại Việt Sử Ký Toàn Thư.*\d{1,4}\s*$',    re.MULTILINE | re.IGNORECASE),
    re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE),
    re.compile(r'^\s*(Đại Việt Sử Ký Toàn Thư|Bản Kỷ|Ngoại Kỷ|Quyển\s+[IVXLC]+)\s*$',
               re.MULTILINE | re.IGNORECASE),
]

BOOK_PAGE_RE = re.compile(
    r'^\s*(\d{1,4})\s+Đại Việt Sử Ký Toàn Thư',
    re.MULTILINE | re.IGNORECASE
)

FOOTNOTE_LINE_RE = re.compile(
    r'^\s{0,9}(\d{1,2})(?:\.|\)|\]|\s{1,4}[A-ZÀ-Ỹ\[])'
)


# def replace_superscripts(text: str) -> str:
#     text = re.sub(r'([a-zA-ZÀ-ỹ\)])(\d+)(?=\s|[,.\!\?;:\)\]\n]|$)', r'\1 [\2]', text)
#     return text

def replace_superscripts(text: str) -> str:
    text = re.sub(r'([a-zA-ZÀ-ỹ\)\.])(\d+)(?=\s|[,.\!\?;:\)\]\n]|$)', r'\1 [\2]', text)
    return text

# header and footer
def remove_headers(text: str) -> str:
    for p in HEADER_FOOTER_PATTERNS:
        text = p.sub('', text)
    return text


def extract_book_page_number(text: str):

    m = BOOK_PAGE_RE.search(text)

    if m:
        return int(m.group(1))

    return None

def normalize_whitespace(text: str) -> str:
    text = re.sub(r'[ \t]+\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'(?<![.!?;:])\n(?!\n)', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()



# ---  TÌM VỊ TRÍ ĐƯỜNG KẺ BẰNG OPENCV ---
def find_footnote_line_y(pdf_page, page_idx):
    """
    Chuyển trang PDF thành ảnh, dùng OpenCV quét tìm đường nằm ngang.
    Trả về tỷ lệ phần trăm vị trí Y trên trang (0.0 đến 1.0) để map với pdfplumber.
    """
    try:
        # Chuyển trang PDF cụ thể thành ảnh (DPI=150 là đủ tốt và nhanh)
        images = convert_from_path(PDF_PATH, first_page=page_idx, last_page=page_idx, dpi=150)
        if not images:
            return None

        open_cv_image = np.array(images[0])
        img = open_cv_image[:, :, ::-1].copy() # RGB to BGR
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        h_img, w_img = gray.shape

        # Nhận diện đường thẳng nằm ngang
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
        detect_horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)

        cnts = cv2.findContours(detect_horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = cnts[0] if len(cnts) == 2 else cnts[1]

        for c in cnts:
            x, y, w, h = cv2.boundingRect(c)
            # Lọc đường kẻ đủ dài (ở đây lớn hơn 15% độ rộng trang ảnh)
            w_ratio = w / w_img

            # Kiểm tra xem tỷ lệ đó có nằm trong khoảng từ 0.15 đến 0.6 không
            if w_ratio >= 0.15 and (h_img / 2) <= y < (h_img * 0.99):
                # print(f"Tìm thấy đường kẻ phù hợp! Tỷ lệ rộng: {w_ratio:.2f}, Vị trí Y: {y} (Thuộc nửa dưới trang)")
                return y / h_img
    except Exception as e:
        pass
    return None


def extract_footnotes_by_layout(page, y_ratio) -> tuple[str, dict]:
    """
    Dựa vào tỷ lệ Y của đường kẻ, cắt không gian trang thành 2 nửa:
    Nửa trên -> Main Text, Nửa dưới -> Footnote Text
    """
    height = page.height
    width = page.width

    # Tính toán tọa độ thực tế trên đối tượng PDF (Bỏ qua 5 pixel nhiễu xung quanh đường kẻ)
    split_y = height * y_ratio

    # Cắt phần trên: từ y=0 đến y=split_y - 5
    main_bbox = (0, 0, width, split_y - 5)
    main_area = page.within_bbox(main_bbox)
    main_text = main_area.extract_text(x_tolerance=3, y_tolerance=3) if main_area else ""

    # Cắt phần dưới: từ y=split_y + 5 đến hết trang
    footer_bbox = (0, split_y + 5, width, height)
    footer_area = page.within_bbox(footer_bbox)
    footer_text = footer_area.extract_text(x_tolerance=3, y_tolerance=3) if footer_area else ""

    # Sử dụng lại logic phân tách từng footnote cũ dựa trên chuỗi chữ của footer_text
    footnote_dict = {}
    if footer_text.strip():
        footer_text = re.sub(r'\b\d{2}(?=\n)', r'\g<0> ', footer_text)
        # # Sau đó xóa dấu \n thừa ngay sau khoảng trắng đó để kéo dòng dưới lên
        footer_text = re.sub(r'(\b\d{2} )\n', r'\1', footer_text)
        lines = footer_text.split('\n')
        print(lines)
        # lines = footer_text
        current_key = None
        current_parts = []

        for line in lines:
            m = FOOTNOTE_LINE_RE.match(line)
            if m:
                if current_key is not None:
                    footnote_dict[current_key] = ' '.join(p.strip() for p in current_parts if p.strip())
                current_key = m.group(1)
                content_line = re.sub(r'^\s*\d{1,2}\s{1,4}', '', line).strip()
                current_parts = [content_line]
            elif current_key is not None:
                current_parts.append(line)

        if current_key is not None:
            footnote_dict[current_key] = ' '.join(p.strip() for p in current_parts if p.strip())

    return main_text, footnote_dict


def clean_page_v2(page, page_idx, raw_text: str) -> tuple[str, dict]:
    if not raw_text:
        return "", {}

    #  Thử tìm đường kẻ cắt trang bằng OpenCV
    y_ratio = find_footnote_line_y(page, page_idx)

    if y_ratio is not None:
        # Nếu tìm thấy đường kẻ: Cắt đôi trang dựa trên tọa độ địa lý hình ảnh
        text, fn_dict = extract_footnotes_by_layout(page, y_ratio)
    else:
          text = raw_text
          fn_dict = {}


    text = replace_superscripts(text)
    text = re.sub(r'\[\d{1,2}[ab]\]', '', text)
    # print(text)
    text = remove_headers(text)
    text = normalize_whitespace(text)

    return text, fn_dict


def extract_all_pages(pdf_path: str) -> list[dict]:
    # START_PAGE = 430
    # END_PAGE   = 450
    results = []
    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        print(f" Mở PDF: {total} trang")
        for i, page in enumerate(tqdm(pdf.pages, desc="📄 Đọc trang", unit="tr"), start=1):
            # if i < START_PAGE:
            #     continue
            # if i >= END_PAGE:
            #     break

            raw = page.extract_text(x_tolerance=3, y_tolerance=3)
            if not raw or len(raw.strip()) < 20:
                continue
        # lấy số trang ở header
            # print(raw)
            book_page = extract_book_page_number(raw)
            # print(f"đang đọc trang {book_page}")

            text, fn_dict = clean_page_v2(page, i, raw)

            if not text or len(text.strip()) < 20:
                continue
            results.append({"page_num": book_page, "text": text, "footnotes": fn_dict})
    return results


def build_full_text(pages: list[dict]) -> tuple[str, dict]:

    parts = []
    page_fn_map = {}

    for page in pages:
        pn = page["page_num"]
        page_fn_map[pn] = page["footnotes"]
        # Đặt marker trước mỗi trang để track page boundary sau split
        parts.append(PAGE_MARKER_FMT.format(page=pn))
        parts.append(page["text"])

    return "".join(parts), page_fn_map


In [4]:

def main():
    print("  XỬ LÝ PDF - ĐẠI VIỆT SỬ KÝ TOÀN THƯ ")

# chuyển từ pdf -> json. Tải về 2 file.
# 1 file chứa các trang đơn lẻ.
# 1 file ghép toàn bộ trang lại với nhau.
    pages = extract_all_pages(PDF_PATH)

    with open("pages_DVSK_data.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    print("Đã lưu pages_data.json")



    full_text, page_fn_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map
    }

    with open("full_DVSK_data.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)

    print(" Đã lưu full_text_data.json")

    from google.colab import files

    files.download("pages_DVSK_data.json")
    files.download("full_DVSK_data.json")


if __name__ == "__main__":
    main()


  XỬ LÝ PDF - ĐẠI VIỆT SỬ KÝ TOÀN THƯ 
 Mở PDF: 154 trang


📄 Đọc trang:   1%|          | 1/154 [00:01<02:43,  1.07s/tr]

['1 Đại Việt sử lược chép đủ tên huý của Lý Huệ Tông là Hạo Sảm (ĐVSL3, 20b).', '2 Bến Triều Đông: bến sông Hồng ở phía đông Thăng Long. cương mục chép là Đông Bộ Đầu và chú là bến Đông Tân sông Nhị Hà.', '3 Nguyên văn: "dĩ Trung Từ vi Thái uý phụ chính, phong Thuận Lưu bá Trần Tự Khánh vi Chương Thành Hầu". Về việc này, Cương', 'mục chép: "Vua bèn phong cho Tự Khánh tước hầu, cho Tô Trung Từ làm Thái uý, phong tước Thuận Lưu bá" (CMCB5, 35b).', 'Đúng ra Lưu Thuận bá là tước của Trần Tự Khánh (như Toàn thư đã chép ở BK4, 26b) và theo Đại Việt sử lược đến năm này', '(Nhâm Thân 1212), ngày Canh Tuất tháng giêng "vua cho Tự Khánh lên tước hầu, tước hiệu là Chương Thành hầu" (ĐVSL3, 24a).', 'Như vậy có thể nhận thấy rằng ở câu của Toàn thư (đã dẫn), soạn giả Cương mục đã đặt nhầm một dấu ngắt đoạn ở sau chữ', '"bá", cho nên mới chép Thuận Lưu bá là tước của Tô Trung Từ.']


📄 Đọc trang:   1%|▏         | 2/154 [00:02<02:39,  1.05s/tr]

['1 Huyện Binh Hợp: chưa rõ ở đâu.', '2 Cửu Liên châu: có lẽ là bãi tả ngạn sông Hồng, gần Cửu Cao, trong đất huyện Văn Giang cũ, nay thuộc huyện Mỹ Văn, tỉnh Hải', 'Hưng.', '3 Quảng Oai: vùng đất ở huyện Chương Mỹ, tỉnh Hà Tây ngày nay.']


...

['1 CMCB 12 chép là Vân Dương Bá.', '2 Tức cửa ải Nam Quan ngày nay.', '3 Một cửa ải gần thị xã Hà Giang ngày nay.']
Đã lưu pages_data.json
 Đã lưu full_text_data.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Khâm Định Việt Sử Thông Giám Cương Mục

In [5]:

import re
import json
import pdfplumber
from tqdm import tqdm

PDF_PATH      = "KDVSTGCMQSQTN.pdf"
OUTPUT_JSON   = "KDVSTGCMQSQTN_chunks.json"
BOOK_NAME     = "Khâm Định Việt Sử Thông Giám Cương Mục"

HEADER_FOOTER_PATTERNS = [
    re.compile(r'^\s*\d{1,4}\s+Khâm Định Việt Sử Thông Giám Cương Mục.*$', re.MULTILINE | re.IGNORECASE),
    re.compile(r'^Khâm Định Việt Sử Thông Giám Cương Mục.*\d{1,4}\s*$',    re.MULTILINE | re.IGNORECASE),
    re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE),
    re.compile(r'^\s*(Khâm Định Việt Sử Thông Giám Cương Mục|Bản Kỷ|Ngoại Kỷ|Quyển\s+[IVXLC]+)\s*$',
               re.MULTILINE | re.IGNORECASE),
]


BOOK_PAGE_RE = re.compile(
    r'^\s*(\d{1,4})\s+Khâm Định Việt Sử Thông Giám Cương Mục.',
    re.MULTILINE | re.IGNORECASE
)


In [6]:
# @title

def main():

    print("  XỬ LÝ PDF - Khâm Định Việt Sử Thông Giám Cương Mục")

    pages = extract_all_pages(PDF_PATH)

    with open("pages_KhamDinh_data.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    full_text, page_fn_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map
    }

    with open("full_KhamDinh_data.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)


    from google.colab import files

    files.download("pages_KhamDinh_data.json")
    files.download("full_KhamDinh_data.json")


if __name__ == "__main__":
    main()

  XỬ LÝ PDF - Khâm Định Việt Sử Thông Giám Cương Mục
 Mở PDF: 165 trang


📄 Đọc trang:   1%|          | 1/165 [00:00<01:53,  1.45tr/s]

['cho công thần, hoàng hậu và công chúa. Ai được phong thang mộc ấp ở nơi nào, thì có quyền sử dụng số thu nhập của ấp ấy chi', 'phí vào mọi việc để bồi dưỡng lòng liêm khiết của mình.', '1 Tức Nội thị chánh thủ. Do chức Hỏa đầu ở đời Thuận Thiên đổi sang. Ở đây là một chức quan ở Nội thị sảnh phụ trách một đội có', 'nhiệm vụ hầu hạ nhà vua.', '2 Người hầu cận trong nội để làm những việc vặt như lấy nước rửa mặt, cầm khăn trầu, v.v...', '3 Đều là các chức quan trong Nội thị sảnh, có nhiệm vụ hầu hạ nhà vua.', '4 Đều là các chức quan trong Nội thị sảnh, có nhiệm vụ hầu hạ nhà vua.', '5 Đều là các chức quan trong Nội thị sảnh, có nhiệm vụ hầu hạ nhà vua.', '6 Ngày trước, ta thường dùng một mảnh vải vuông làm khăn đựng các đồ ăn trầu, cau, vỏ, ống vôi gọi là "khăn trầu".', '7 Các vua Lý.']


📄 Đọc trang:   1%|          | 2/165 [00:01<02:17,  1.19tr/s]

['1 Cha của vua. Đây chỉ Trần Thừa.', '2 Người Long Cương, Sài Vinh là con người anh của Sài Thị, vợ Chu Thái Tổ, đời Ngũ đại (923-959); được làm con nuôi nhà vua,', 'sau Sài Vinh lên nối ngôi Chu Thái Tổ, Vinh làu thông kinh sử, có tài chính trị; khi cầm quyền rồi, lấy được Trần Lũng, dẹp yên', 'Hoài Hữu, oai danh lừng lẫy khắp nơi. Vinh lại sửa lễ nhạc, đặt chế độ, có nhiều chính sách khả quan. Khi mất, miếu hiệu là Chu', 'Thế Tông.', '3 Vương Mãng là cháu Hiếu Nguyên hoàng hậu nhà Hán, sau giết Hán Bình đế, đưa Nhụ Tử Anh lên ngôi được hai năm, rồi cướp', 'ngôi nhà Hán.', '4 Dương Kiên thời Nam Bắc triều. Con gái Dương Kiên là hoàng hậu của Tuyên Đế nhà Haậu Chu (951-959). Sau khi Tuyên đế mất,', 'Dương Kiên bỏ con Tuyên đế là Tĩnh đế, tự lập làm vua, tức là Tùy Văn đế.', '5 Đa Nhĩ Cổn là chú ruột Thanh Thế tổ (Thuận Trị, 1644-1661), phá Lý Tự Thành, dẹp yên kinh đô, đón Thế tổ vào trong quan ải.', 'Khi Thế tổ còn nhỏ, Đa Nhĩ Cổn phải tạm cầm chính quyền, xưng là nhiếp chính vương.'

...

['1 Chỉ bọn Nguyễn Huân.', '2 Lời phê này chúng tôi dịch thật sát với nguyên văn. "Người Nam Kỳ" đây là chỉ một số người đã đầu hàng giặc Pháp hồi Tự Đức.', '3 Ký chế là dịch theo âm Hán - Việt, thực ra dân ở địa phương này gọi là Cấy Chấy.', '4 Nay thuộc tỉnh Phú Thọ.', '5 Nguyên văn chép là "vi tử thủ". Theo chú thích trong Cương mục Chính biên XXXV, 14, vi tử là những người sung vào làm công', 'việc ở nơi quan phủ. Ở đây, có lẽ Trương Phụ chọn những người khoẻ mạnh hùng dũng đem vào dinh thự cho ở xung quanh', 'mình để đề phòng sự bất trắc.', '6 Chức quan coi về quân lính.']
['1 Quyển sổ cần để khảo cứu cho biết tình hình số hộ số khẩu, ruộng đất và thuế lương.']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Việt Sử Toàn Thư


In [7]:

import re
import json
import pdfplumber
from tqdm import tqdm

PDF_PATH      = "VSTT.pdf"
OUTPUT_JSON   = "VSTT_chunks.json"
BOOK_NAME     = "Việt Sử Toàn Thư"


HEADER_FOOTER_PATTERNS = [
    re.compile(r'^\s*\d{1,4}\s+Việt Sử Toàn Thư.*$', re.MULTILINE | re.IGNORECASE),
    re.compile(r'^Việt Sử Toàn Thư.*\d{1,4}\s*$',    re.MULTILINE | re.IGNORECASE),
    re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE),
    re.compile(r'^\s*(Việt Sử Toàn Thư|Bản Kỷ|Ngoại Kỷ|Quyển\s+[IVXLC]+)\s*$',
               re.MULTILINE | re.IGNORECASE),
]

FOOTNOTE_LINE_RE = re.compile(
    r'^\s{0,9}(\d{1,2})(\s{1,4}[A-ZÀ-Ỹ\[])'
)


BOOK_PAGE_RE = re.compile(
    r'^\s*(\d{1,4})\s+Việt Sử Toàn Thư',
    re.MULTILINE | re.IGNORECASE
)






In [8]:

def main():
    print(f"Sách: {BOOK_NAME}")
    pages = extract_all_pages(PDF_PATH)

    with open("pages_VietSu_data.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    full_text, page_fn_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map
    }

    with open("full_VietSu_data.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)


    from google.colab import files

    files.download("pages_VietSu_data.json")
    files.download("full_VietSu_data.json")


if __name__ == "__main__":
    main()

Sách: Việt Sử Toàn Thư
 Mở PDF: 89 trang


📄 Đọc trang:   7%|▋         | 6/89 [00:05<01:25,  1.03s/tr]

['35 Theo ý chúng tôi nhà Trần có dự bị tổng động viên nhưng chưa hề làm việc này cho nên Trần', 'Nhân Tông đã có câu:', 'Cối kê cưu sự, quân tư ký,', 'Hoan, Diễn do tồn thập vạn binh', 'nghĩa là Câu Tiễn xưa kia thua Ngô còn lại năm ngàn giáp, thuẫn trú đầu ở Cối Kê, thế mà sau còn', 'khôi phục được giang sơn, tiêu diệt được địch quốc, huống hồ ta còn mười vạn quân Thanh Nghệ chưa', 'gọi tới.']


📄 Đọc trang:   9%|▉         | 8/89 [00:07<01:14,  1.09tr/s]

['36 Nguyên văn "Cùng dân bất cập giá hứa diện túc ư nhân".']


...

📄 Đọc trang: 100%|██████████| 89/89 [00:52<00:00,  1.69tr/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Vương Triều Trần (1226-1400) - Vũ Văn Quân

In [11]:
import re
import json
from tqdm import tqdm
import pdfplumber

PDF_PATH      = "VTT.pdf"
OUTPUT_JSON   = "VTT_chunks.json"
BOOK_NAME     = "Vương Triều Trần (1226-1400)"

HEADER_FOOTER_PATTERNS = [
    re.compile(r'^\s*[A-ZÀ-Ỹa-zà-ỹ\s]+\s*\(\s*Chủ\s+biên\s*\)\s*$', re.IGNORECASE | re.MULTILINE),
    re.compile(r'^\s*VƯƠNG\s+TRIỀU\s+[A-ZÀ-Ỹa-zà-ỹ\s]+\s*\(\s*\d{4}\s*-\s*\d{4}\s*\)\s*$', re.IGNORECASE |  re.MULTILINE),
    re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE |  re.IGNORECASE),
]


FOOTNOTE_LINE_RE = re.compile(
    r'^\s*(\d{1,2})(?:\.|\)|\]|\s*[A-ZÀ-Ỹ\[])'
)

def split_double_annotations(text: str) -> str:
    # Định dạng: Đầu dòng, Số 1, Dấu phẩy, Số 2, Dấu chấm, Khoảng trắng, và toàn bộ Nội dung phía sau
    pattern = r'^(?P<num1>\d+),(?P<num2>\d+)[\.\,]\s*(?P<content>.*)$'
    def replace_callback(match):
        n1 = match.group('num1')
        n2 = match.group('num2')
        content = match.group('content')
        print(f"hehe {n1}. {content}\n{n2}. {content}")
        return f"{n1}. {content}\n{n2}. {content}"

    # Áp dụng trên từng dòng độc lập (flags=re.MULTILINE)
    return re.sub(pattern, replace_callback, text, flags=re.MULTILINE)

def extract_footnotes_by_layout(page, y_ratio) -> tuple[str, dict]:
    """
    Dựa vào tỷ lệ Y của đường kẻ, cắt không gian trang thành 2 nửa:
    Nửa trên -> Main Text, Nửa dưới -> Footnote Text
    """
    height = page.height
    width = page.width

    # Tính toán tọa độ thực tế trên đối tượng PDF (Bỏ qua 5 pixel nhiễu xung quanh đường kẻ)
    split_y = height * y_ratio

    # Cắt phần trên: từ y=0 đến y=split_y - 5
    main_bbox = (0, 0, width, split_y - 5)
    main_area = page.within_bbox(main_bbox)
    main_text = main_area.extract_text(x_tolerance=3, y_tolerance=3) if main_area else ""

    # Cắt phần dưới: từ y=split_y + 5 đến hết trang
    footer_bbox = (0, split_y + 5, width, height)
    footer_area = page.within_bbox(footer_bbox)
    footer_text = footer_area.extract_text(x_tolerance=3, y_tolerance=3) if footer_area else ""

    footnote_dict = {}
    if footer_text.strip():
        footer_text = re.sub(r'\b\d{2}(?=\n)', r'\g<0> ', footer_text)
        # Sau đó xóa dấu \n thừa ngay sau khoảng trắng đó để kéo dòng dưới lên
        footer_text = re.sub(r'(\b\d{2} )\n', r'\1', footer_text)

        footer_text = split_double_annotations(footer_text)
        footer_text =  re.sub(r'(?<=[a-zà-ỹ])\n(?=[a-zà-ỹ])', ' ', footer_text)
        lines = footer_text.split('\n')

        current_key = None
        current_parts = []

        for line in lines:
            line = re.sub(r'^\s*\d{2,3}\s*$\n?', '', line, flags=re.MULTILINE)

            m = FOOTNOTE_LINE_RE.match(line)
            if m:
                if current_key is not None:
                    footnote_dict[current_key] = ' '.join(p.strip() for p in current_parts if p.strip())
                current_key = m.group(1)
                content_line = re.sub(r'^\s*\d{1,2}[,.]?\s*', '', line).strip()
                current_parts = [content_line]
            elif current_key is not None:
                current_parts.append(line)

        if current_key is not None:
            footnote_dict[current_key] = ' '.join(p.strip() for p in current_parts if p.strip())

    return main_text, footnote_dict


def extract_all_pages(pdf_path: str) -> list[dict]:
    START_PAGE = 9
    END_PAGE   = 769
    results = []
    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        print(f" Mở PDF: {total} trang")
        for i, page in enumerate(tqdm(pdf.pages, desc="📄 Đọc trang", unit="tr"), start=1):
            if i < START_PAGE:
                continue
            if i >= END_PAGE:
                break
            raw = page.extract_text(x_tolerance=3, y_tolerance=3)
            if not raw or len(raw.strip()) < 20:
                continue
        # # lấy số trang ở header
        #     book_page = extract_book_page_number(raw)

            text, fn_dict = clean_page_v2(page, i, raw)

            if not text or len(text.strip()) < 20:
                continue
            results.append({"page_num": i-2, "text": text, "footnotes": fn_dict})
    return results

In [12]:

def main():
    print(f"Sách: {BOOK_NAME}")
    pages = extract_all_pages(PDF_PATH)

    with open("pages_VTT_data1.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    full_text, page_fn_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map
    }

    with open("full_VTT_data1.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)


    from google.colab import files

    files.download("pages_VTT_data1.json")
    files.download("full_VTT_data1.json")


if __name__ == "__main__":
    main()

Sách: Vương Triều Trần (1226-1400)
 Mở PDF: 806 trang


📄 Đọc trang:   1%|          | 9/806 [00:00<00:46, 16.96tr/s]

hehe 271. 
276. 


📄 Đọc trang:   1%|▏         | 11/806 [00:01<02:25,  5.45tr/s]

hehe 1. Việt sử lược, Sđd, tr.166, 165.
2. Việt sử lược, Sđd, tr.166, 165.


...

📄 Đọc trang:  95%|█████████▌| 768/806 [08:38<00:25,  1.48tr/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>